## Section 2a: Shared Text Preprocessing Pipeline

All four application features rely on a common preprocessing pipeline implemented in
`core/preprocessing.py`. The pipeline normalises raw review text into a consistent
token sequence before any feature extraction or model inference.

### 5-Step Pipeline (`preprocess_text`)

| Step | Operation | Detail |
|---|---|---|
| 1 | Lowercase + collocation substitution | Replace known bigrams with hyphenated form (e.g. `"dry skin"` → `"dry-skin"`) so the pair is treated as one token |
| 2 | Regex tokenisation | Pattern `r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"` — preserves hyphenated and apostrophe tokens |
| 3 | Short word removal | Drop tokens with `len < 2` |
| 4 | Stopword removal | Remove the 570-word list from `stopwords_en.txt` |
| 5 | Lemmatisation | `nltk.WordNetLemmatizer` — reduces inflected forms to base lemma |

### Embedding Transformers

Two `sklearn`-compatible transformers convert preprocessed text to dense vectors:

- **`UnweightedVectorTransformer`** — returns the simple mean of the GloVe vectors for all
  in-vocabulary tokens (falls back to a zero vector if no token matches).
  Used by Model A (text) and both text/title channels of Model C.

- **`WeightedVectorTransformer`** — computes a TF-IDF weighted mean of word vectors using
  an internally fitted `TfidfVectorizer`.
  Defined for completeness; the current ensemble uses unweighted vectors.

### How Preprocessing Connects to Each Feature

| Task | Where preprocessing runs |
|---|---|
| **Task 1 — Search** | The user's search query is preprocessed to compute a GloVe mean embedding for semantic similarity |
| **Task 2 — Write Review** | New review text and title are preprocessed before being passed to all three classifiers |
| **Task 3 — Recommendation** | Product vectors were built from preprocessed review text during offline training |
| **Task 4 — Sentiment Insights** | Key-term extraction operates on preprocessed tokens; sentiment scoring uses raw rating values |

In [ ]:
import re
from nltk.stem import WordNetLemmatizer

# Tokenisation pattern: matches alpha tokens, including hyphenated/apostrophe forms
# e.g. "anti-aging", "can't"  →  kept as single tokens
REGEX_PATTERN = r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"


def preprocess_text(text, colloc_dict, stop_words, lemmatizer=None):
    """
    Preprocess raw review text through the 5-step pipeline.

    Returns a space-separated string of cleaned tokens ready for
    vectorisation (BoW, GloVe mean, etc.).
    """
    if lemmatizer is None:
        lemmatizer = WordNetLemmatizer()

    # Step 1: lowercase + replace known bigrams with hyphenated tokens
    text = str(text).lower()
    for bigram, replacement in colloc_dict.items():
        if bigram in text:
            text = text.replace(bigram, replacement)

    # Step 2: regex tokenisation
    tokens = [m.group(0) for m in re.finditer(REGEX_PATTERN, text)]

    # Step 3: remove short tokens (single characters add noise)
    tokens = [t for t in tokens if len(t) >= 2]

    # Step 4: stopword removal
    tokens = [t for t in tokens if t not in stop_words]

    # Step 5: lemmatise to canonical form
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer


class UnweightedVectorTransformer(BaseEstimator, TransformerMixin):
    """
    sklearn transformer: text → 300-dim mean GloVe vector (no weighting).

    For each document, computes the arithmetic mean of the GloVe vectors
    of all in-vocabulary tokens.  Used in Model A (text only) and both
    text/title channels of Model C.
    """

    def __init__(self, embeddings_dict):
        self.embeddings_dict = embeddings_dict
        self.dim = len(next(iter(embeddings_dict.values()))) if embeddings_dict else 300

    def fit(self, X, y=None):
        return self   # stateless — no fitting required

    def transform(self, X):
        result = []
        for text in X:
            tokens = str(text).split()
            vecs   = [self.embeddings_dict[w] for w in tokens if w in self.embeddings_dict]
            result.append(np.mean(vecs, axis=0) if vecs else np.zeros(self.dim))
        return np.array(result)


class WeightedVectorTransformer(BaseEstimator, TransformerMixin):
    """
    sklearn transformer: text → TF-IDF weighted mean GloVe vector.

    Fits a TfidfVectorizer on the training corpus, then at transform time
    uses each token's TF-IDF weight as the coefficient for its GloVe vector.
    """

    def __init__(self, embeddings_dict):
        self.embeddings_dict = embeddings_dict
        self.tfidf = None
        self.dim   = len(next(iter(embeddings_dict.values()))) if embeddings_dict else 300

    def fit(self, X, y=None):
        self.tfidf = TfidfVectorizer()
        self.tfidf.fit(X)
        return self

    def transform(self, X):
        tfidf_matrix = self.tfidf.transform(X)
        vocab        = self.tfidf.vocabulary_
        result       = []
        for row_num, text in enumerate(X):
            tokens = str(text).split()
            vecs, wts = [], []
            for w in tokens:
                if w in self.embeddings_dict and w in vocab:
                    vecs.append(self.embeddings_dict[w])
                    wts.append(tfidf_matrix[row_num, vocab[w]])
            if vecs and sum(wts) > 0:
                result.append(np.average(vecs, axis=0, weights=wts))
            else:
                result.append(np.zeros(self.dim))
        return np.array(result)